In [17]:
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from transformers import SegformerForSemanticSegmentation, SegformerConfig
import os
import numpy as np
from tqdm import tqdm
import json

# 数据集类
class TreeTrunkDataset(Dataset):
    def __init__(self, image_dir, mask_dir, json_dir, transform=None, target_size=(1024, 1024)):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.json_dir = json_dir
        self.transform = transform
        self.target_size = target_size  # 目标尺寸
        self.images = [img for img in os.listdir(image_dir) if img.lower().endswith(('.jpg', '.jpeg', '.png'))]
        self.images = [img for img in self.images if os.path.exists(os.path.join(mask_dir, img.replace(".JPG", ".png").replace(".jpg", ".png").replace(".jpeg", ".png")))]
        self.images = [img for img in self.images if os.path.exists(os.path.join(json_dir, img.replace(".JPG", ".json").replace(".jpg", ".json").replace(".jpeg", ".json")))]

        # 添加调试信息
        if len(self.images) == 0:
            print("未找到匹配的图像、掩码或 JSON 文件，请检查目录和文件扩展名。")
        else:
            print(f"找到 {len(self.images)} 个匹配的图像、掩码和 JSON 文件。")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.images[idx].replace(".JPG", ".png").replace(".jpg", ".png").replace(".jpeg", ".png"))
        json_path = os.path.join(self.json_dir, self.images[idx].replace(".JPG", ".json").replace(".jpg", ".json").replace(".jpeg", ".json"))
        
        # 读取图像
        image = Image.open(img_path).convert("RGB")  # 使用 Pillow 打开图像
        mask = Image.open(mask_path).convert('L')  # 转换为灰度图像
        mask = np.array(mask)  # 转换为numpy数组
        
        # 读取 JSON 文件
        with open(json_path, 'r') as f:
            json_data = json.load(f)

        # 统一图像和掩码的大小
        image = image.resize(self.target_size)
        mask = Image.fromarray(mask).resize(self.target_size, Image.NEAREST)
        
        # 转换为 Tensor
        if self.transform:
            image = self.transform(image)

        # 确保掩码值在0和1之间
        mask = np.clip(mask, 0, 1)
        mask = torch.tensor(mask, dtype=torch.long)
        
        return image, mask, json_data  # 返回 image, mask 和 json_data

# 自定义 collate_fn，确保每个批次中的数据大小一致
def collate_fn(batch):
    images, masks, json_data = zip(*batch)

    # 将所有图像堆叠成一个批次
    images = torch.stack(images, dim=0)
    
    # 将所有掩码堆叠成一个批次
    masks = torch.stack(masks, dim=0)

    return images, masks

# 设置路径
image_dir = "D:/2024/paper2/model/input_images2/cihuai"
mask_dir = "D:/2024/paper2/model/output_masks/cihuai"
json_dir = "D:/2024/paper2/model/sorted_JSON/cihuai"

# 数据增强与预处理
transform = transforms.Compose([
    transforms.ToTensor(),  # 转换为 Tensor
])

# 数据集和数据加载器
dataset = TreeTrunkDataset(image_dir=image_dir, mask_dir=mask_dir, json_dir=json_dir, transform=transform)

# 检查数据集是否为空
if len(dataset) == 0:
    raise ValueError("数据集为空，请检查图像和掩码文件是否匹配且存在。")

# 使用自定义的 collate_fn 确保每个批次的大小一致
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

# 初始化模型
config = SegformerConfig()
config.num_labels = 2  # 假设只有背景和前景两类
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SegformerForSemanticSegmentation(config).to(device)

# 定义损失函数和优化器
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

# 自定义中心加权损失
class WeightedCrossEntropyLoss(nn.Module):
    def __init__(self, weight_center=2.0):
        super(WeightedCrossEntropyLoss, self).__init__()
        self.weight_center = weight_center
        self.ce_loss = nn.CrossEntropyLoss(reduction='none')

    def forward(self, inputs, targets):
        # 确保 inputs 和 targets 的维度一致
        if inputs.shape[2:] != targets.shape:
            targets = targets.unsqueeze(1)
            targets = torch.nn.functional.interpolate(targets.float(), size=inputs.shape[2:], mode='nearest').squeeze(1).long()
        
        # 计算基本交叉熵损失
        loss = self.ce_loss(inputs, targets)

        # 中心区域权重：仅对纵向范围中心加权，横向不变
        _, _, h, w = inputs.shape
        y_center, x_center = h // 2, w // 2
        weight_mask = torch.ones_like(targets, dtype=torch.float32)
        weight_mask[y_center - h // 4:y_center + h // 4, :] = self.weight_center  # 仅纵向范围

        # 应用权重
        weighted_loss = loss * weight_mask.to(inputs.device)
        return weighted_loss.mean()

criterion = WeightedCrossEntropyLoss(weight_center=5.0)

# 训练循环
epochs = 50
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    with tqdm(total=len(dataloader), desc=f"Epoch {epoch+1}/{epochs}") as pbar:
        for images, masks in dataloader:  # 加载图像和掩码
            images, masks = images.to(device), masks.to(device)

            # 前向传播
            outputs = model(images).logits
            # 调整输出大小以匹配掩码大小
            outputs = torch.nn.functional.interpolate(outputs, size=(masks.shape[1], masks.shape[2]), mode='bilinear', align_corners=False)

            # 计算损失
            loss = criterion(outputs, masks)

            # 反向传播和优化
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            pbar.update(1)  # 更新 tqdm 进度条
            pbar.set_postfix({"Loss": f"{loss.item():.4f}"})  # 显示当前损失

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(dataloader):.4f}")

# 保存模型
save_dir = "D:/2024/paper2/model/Model"
os.makedirs(save_dir, exist_ok=True)

# 保存完整模型
model_save_path = os.path.join(save_dir, "segformer_full_model.pth")
torch.save(model, model_save_path)
print(f"完整模型已保存到 {model_save_path}")

# 保存模型权重
weights_save_path = os.path.join(save_dir, "segformer_weights.pth")
torch.save(model.state_dict(), weights_save_path)
print(f"模型权重已保存到 {weights_save_path}")


# 保存配置文件
config_save_path = os.path.join(save_dir, "config.json")
config.save_pretrained(save_dir)
print(f"配置文件已保存到 {config_save_path}")

# 保存检查点文件
checkpoint_save_path = os.path.join(save_dir, "checkpoint.pth")
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config": config.to_dict(),
    "epoch": epochs,
}, checkpoint_save_path)
print(f"检查点文件已保存到 {checkpoint_save_path}")

print("训练完成并保存所有文件！")

找到 70 个匹配的图像、掩码和 JSON 文件。


Epoch 1/50: 100%|██████████| 18/18 [01:41<00:00,  5.64s/it, Loss=0.4365]


Epoch [1/50], Loss: 0.5624


Epoch 2/50: 100%|██████████| 18/18 [01:59<00:00,  6.61s/it, Loss=0.3582]


Epoch [2/50], Loss: 0.3910


...

Epoch [48/50], Loss: 0.0387


Epoch 49/50: 100%|██████████| 18/18 [03:09<00:00, 10.54s/it, Loss=0.0310]


Epoch [49/50], Loss: 0.0389


Epoch 50/50: 100%|██████████| 18/18 [03:08<00:00, 10.46s/it, Loss=0.0319]

Epoch [50/50], Loss: 0.0394
完整模型已保存到 D:/2024/paper2/model/Model\segformer_full_model.pth
模型权重已保存到 D:/2024/paper2/model/Model\segformer_weights.pth
配置文件已保存到 D:/2024/paper2/model/Model\config.json
检查点文件已保存到 D:/2024/paper2/model/Model\checkpoint.pth
训练完成并保存所有文件！


In [22]:
import torch
import os
from PIL import Image
from torchvision import transforms
from transformers import SegformerForSemanticSegmentation
import numpy as np

# 设置路径
image_dir = "D:/2024/paper2/model/test/cihuai"  # 验证图像文件夹
output_mask_dir = "D:/2024/paper2/model/test_masks/cihuai"  # 输出掩码文件夹
model_dir = "D:/2024/paper2/model/Model/cihuai"  # 模型文件夹

# 加载完整模型
model_path = os.path.join(model_dir, "segformer_full_model.pth")
model = torch.load(model_path)  # 直接加载完整模型
model.eval()  # 设置为评估模式

# 数据预处理（与训练时相同）
transform = transforms.Compose([
    transforms.ToTensor(),  # 转换为 Tensor
])

# 创建输出文件夹（如果不存在）
os.makedirs(output_mask_dir, exist_ok=True)

# 验证集图像文件名
image_filenames = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

# 推理过程
with torch.no_grad():  # 在推理时不需要计算梯度
    for filename in image_filenames:
        img_path = os.path.join(image_dir, filename)
        
        # 读取图像
        image = Image.open(img_path).convert("RGB")
        
        # 保存原始尺寸
        original_size = image.size
        
        # 统一尺寸（假设训练时输入尺寸是1024x1024）
        target_size = (1024, 1024)  # 如果你的验证图像不是1024x1024，可能需要调整此处
        image_resized = image.resize(target_size)

        # 进行预处理
        image_tensor = transform(image_resized).unsqueeze(0).to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

        # 模型推理
        outputs = model(image_tensor).logits

        # 获取预测的类别标签（背景=0, 树干=1）
        predicted_mask = torch.argmax(outputs, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

        # 将输出的掩码转换为 0 和 255（黑色背景，白色树干）
        predicted_mask = predicted_mask * 255  # 0 -> 0, 1 -> 255

        # 将预测掩码调整回原始输入图像的大小
        predicted_mask_resized = Image.fromarray(predicted_mask)
        predicted_mask_resized = predicted_mask_resized.resize(original_size, Image.NEAREST)

        # 保存掩码图像
        mask_filename = os.path.join(output_mask_dir, filename.replace(".jpg", ".png").replace(".jpeg", ".png").replace(".png", ".png"))
        predicted_mask_resized.save(mask_filename)

        print(f"Saved mask for {filename} to {mask_filename}")

print("推理完成，所有掩码已保存！")


C:\Users\admin\AppData\Local\Temp\ipykernel_14196\2477272780.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(model_path)  # 直接加载完整模型


Saved mask for 01141010087000_01.jpg to D:/2024/paper2/model/test_masks/cihuai\01141010087000_01.png
Saved mask for 01141010087000_03.jpg to D:/2024/paper2/model/test_masks/cihuai\01141010087000_03.png
Saved mask for 01141010087000_04.jpg to D:/2024/paper2/model/test_masks/cihuai\01141010087000_04.png
Saved mask for 07080070013000_04.jpg to D:/2024/paper2/model/test_masks/cihuai\07080070013000_04.png
Saved mask for 09090010025000_02.jpg to D:/2024/paper2/model/test_masks/cihuai\09090010025000_02.png
Saved mask for 09090010025000_03.jpg to D:/2024/paper2/model/test_masks/cihuai\09090010025000_03.png
Saved mask for 09090010025000_04.jpg to D:/2024/paper2/model/test_masks/cihuai\09090010025000_04.png
Saved mask for 09090010025000_05.jpg to D:/2024/paper2/model/test_masks/cihuai\09090010025000_05.png
Saved mask for 09090180007000_04.jpg to D:/2024/paper2/model/test_masks/cihuai\09090180007000_04.png
Saved mask for 09090180007000_05.jpg to D:/2024/paper2/model/test_masks/cihuai\090901800070

In [3]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# 设置路径
image_dir = "D:/2024/paper2/model/test/cihuai"  # 验证图像文件夹
mask_dir = "D:/2024/paper2/model/test_masks/cihuai"  # 单通道掩码文件夹
output_dir = "D:/2024/paper2/model/test_comparison/cihuai"  # 输出图片文件夹

# 创建输出文件夹（如果不存在）
os.makedirs(output_dir, exist_ok=True)

# 获取图像文件名
image_filenames = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

# 处理每个图像
for filename in image_filenames:
    # 构造图像路径和掩码路径
    img_path = os.path.join(image_dir, filename)
    mask_path = os.path.join(mask_dir, filename.replace(".jpg", ".png").replace(".jpeg", ".png").replace(".png", ".png"))

    # 读取验证图像和掩码图像
    image = Image.open(img_path).convert("RGB")  # 读取并转换为 RGB 图像
    mask = Image.open(mask_path).convert("L")  # 读取掩码并转换为灰度图像（L 模式）

    # 将掩码图像转换为 numpy 数组（0 表示背景，255 表示树干）
    mask_array = np.array(mask)

    # 将掩码值转换为 0 或 1（0 -> 背景，1 -> 树干）
    mask_binary = (mask_array > 127).astype(int)  # 将大于127的像素值转为 1，小于等于127的为 0

    # 将原始图像和掩码图像叠加
    image_array = np.array(image)

    # 使用掩码区域将原始图像覆盖成不同的颜色（比如将树干区域用红色标出）
    # 红色区域表示模型预测的树干区域
    image_with_mask = image_array.copy()
    image_with_mask[mask_binary == 1] = [255, 0, 0]  # 将树干区域标记为红色 [255, 0, 0]

    # 将原始图像与掩码图像并排显示
    # 设置高分辨率图像
    fig, axes = plt.subplots(1, 2, figsize=(16, 8), dpi=200)  # 设置更大的图像大小和更高的 DPI

    # 显示原始图像
    axes[0].imshow(image)
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    # 显示叠加图像
    axes[1].imshow(image_with_mask)
    axes[1].set_title("Image with Predicted Mask")
    axes[1].axis("off")

    # 保存结果图像
    comparison_filename = os.path.join(output_dir, filename.replace(".jpg", "_comparison.png").replace(".jpeg", "_comparison.png").replace(".png", "_comparison.png"))
    plt.savefig(comparison_filename, dpi=200)  # 保存为高分辨率图像
    plt.close()

    print(f"Saved comparison image for {filename} to {comparison_filename}")

print("所有比较图像已保存！")


Saved comparison image for 01141010087000_01.jpg to D:/2024/paper2/model/test_comparison/cihuai\01141010087000_01_comparison_comparison.png
Saved comparison image for 01141010087000_03.jpg to D:/2024/paper2/model/test_comparison/cihuai\01141010087000_03_comparison_comparison.png
Saved comparison image for 01141010087000_04.jpg to D:/2024/paper2/model/test_comparison/cihuai\01141010087000_04_comparison_comparison.png
Saved comparison image for 07080070013000_04.jpg to D:/2024/paper2/model/test_comparison/cihuai\07080070013000_04_comparison_comparison.png
Saved comparison image for 09090010025000_02.jpg to D:/2024/paper2/model/test_comparison/cihuai\09090010025000_02_comparison_comparison.png
Saved comparison image for 09090010025000_03.jpg to D:/2024/paper2/model/test_comparison/cihuai\09090010025000_03_comparison_comparison.png
Saved comparison image for 09090010025000_04.jpg to D:/2024/paper2/model/test_comparison/cihuai\09090010025000_04_comparison_comparison.png
Saved comparison ima